In [2]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher, tqdm
StreamlitPatcher().jupyter()  # register streamlit with jupyter-compatible wrappers
import sys; sys.path.append('..')
from osp import *
pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 20

In [3]:
df_preds, df_feats, d_preds = get_preds_feats()

In [4]:
groups_train = [
    ('ELH',"journal in ['ELH']"),
    ('Critical Inquiry',"journal in ['Critical Inquiry']"),
]

In [32]:
df_smpl = get_balanced_slice_sample(groups_train, sample_size=None, balance=True)
df_smpl.journal.value_counts()

journal
ELH                 1401
Critical Inquiry    1401
Name: count, dtype: int64

In [33]:
id = df_smpl.sample(1).slice_id.values[0]
id

'lit/2872685__03'

In [200]:

def get_diff_rows(df_smpl_feats):
    new_rows = []
    for feat,feat_df in df_smpl_feats.groupby('feat'):
        feat2means = feat_df.groupby('target').mean(numeric_only=True)
        new_row = {
            'feat': feat,
        }
        for target2 in feat2means.index[1:]:
            target1 = feat2means.index[0]
            target = f'{target1} - {target2}'
            z1 = feat2means.loc[target1,'z']
            z2 = feat2means.loc[target2,'z']
            z_diff = z1 - z2
            
            raw1 = feat2means.loc[target1,'raw']
            raw2 = feat2means.loc[target2,'raw']
            raw_diff = raw1 - raw2
            
            new_row = {
                'feat': feat,
                'target': f'{target2_i} - {target1_i}',
                'z': z_diff,
                'raw': raw_diff,
            }
            new_rows.append(new_row)
    feat2rank = df_smpl_feats.groupby('feat').feat_rank.first().to_dict()
    new_rows_df = pd.DataFrame(new_rows)
    new_rows_df['feat_rank'] = new_rows_df['feat'].map(feat2rank)
    new_rows_df = new_rows_df.sort_values('feat_rank')
    new_rows_df['feat_diff_rank'] = new_rows_df['z'].abs().rank(method='dense', ascending=False).apply(int)
    new_rows_df['z_abs'] = new_rows_df['z'].abs()
    return new_rows_df.sort_values('feat_diff_rank')

In [201]:
def get_balanced_slice_sample_feats(groups_train_or_df_smpl, sample_size=None, balance=True, with_diff_rows=True):
    if isinstance(groups_train_or_df_smpl, pd.DataFrame):
        df_smpl = groups_train_or_df_smpl
    else:
        df_smpl = get_balanced_slice_sample(
            groups_train_or_df_smpl, 
            sample_size=sample_size, 
            balance=balance
        )

    df_all_feats_z = get_all_feats(normalize=True)
    df_all_feats_raw = get_all_feats(normalize=False)

    valid_ids = set(df_smpl.slice_id) & set(df_all_feats_z.index)
    df_smpl_valid = df_smpl[df_smpl.slice_id.isin(valid_ids)].set_index('slice_id')
    df_smpl_feats_z = df_all_feats_z.loc[df_smpl_valid.index]
    df_smpl_feats_z['_target'] = df_smpl_valid['_target']
    
    df_smpl_feats_raw = df_all_feats_raw.loc[df_smpl_valid.index]

    o = []
    for feat in df_smpl_feats_z.columns:
        for slice_id in df_smpl_feats_z.index:        
            target = df_smpl_feats_z.loc[slice_id, '_target']
            if feat and feat[0]!='_':
                d = {
                    'slice_id': slice_id,
                    'target': target,
                    'feat': feat,
                    'z': df_smpl_feats_z.loc[slice_id, feat],
                    'raw': df_smpl_feats_raw.loc[slice_id, feat],
                }
                o.append(d)
        
    odf = pd.DataFrame(o)
    odf = odf.groupby(['feat','target']).mean(numeric_only=True).reset_index()
    odf['z_abs'] = odf['z'].abs()
    feat2max_abs_z = odf.groupby('feat').z_abs.max()
    ranked_feat2max_abs_z = feat2max_abs_z.rank(method='dense', ascending=False).apply(int)    
    odf['feat_rank'] = odf['feat'].map(ranked_feat2max_abs_z)

    if with_diff_rows:
        df_diff_rows = get_diff_rows(odf)
        feat2diff_rank = df_diff_rows.groupby('feat').feat_diff_rank.first().to_dict()
        odf['feat_diff_rank'] = odf['feat'].map(feat2diff_rank)
        odf = pd.concat([
            odf.set_index(['feat','target']), 
            df_diff_rows.set_index(['feat','target'])
        ])
    return odf.sort_values(['feat_diff_rank', 'z'],ascending=[True, False])#.set_index(['feat','target'])

In [ ]:
df_smpl_feats = get_balanced_slice_sample_feats(groups_train, with_diff_rows=True)
df_smpl_feats

z        raw     z_abs  \
feat             target                                                  
pos_POS          ELH                     1.184879  11.264025  1.184879   
                 Critical Inquiry        0.092602   5.804871  0.092602   
                 ELH - Critical Inquiry -1.092276  -5.459154  1.092276   
deprel_nmod:poss ELH                     1.220786  28.965105  1.220786   
                 Critical Inquiry        0.210242  19.362032  0.210242   
...                                           ...        ...       ...   
phrase_$         ELH - Critical Inquiry  0.000000   0.000000  0.000000   
                 Critical Inquiry       -0.009382   0.000000  0.009382   
                 ELH                    -0.009382   0.000000  0.009382   
phrase_NAC       Critical Inquiry       -0.012079   0.000000  0.012079   
                 ELH                    -0.012079   0.000000  0.012079   

                                         feat_rank  feat_diff_rank  
feat             target                                             
pos_POS          ELH                             2               1  
                 Critical Inquiry                2               1  
                 ELH - Critical Inquiry          2               1  
deprel_nmod:poss ELH                             1               2  
                 Critical Inquiry                1               2  
...                                            ...             ...  
phrase_$         ELH - Critical Inquiry        131             130  
                 Critical Inquiry              131             130  
                 ELH                           131             130  
phrase_NAC       Critical Inquiry              130             130  
                 ELH                           130             130  

[393 rows x 5 columns]

feat                  target         z  \
feat                                                                       
deprel_acl       41         deprel_acl  ELH - Critical Inquiry  0.159982   
                 82         deprel_acl        Critical Inquiry -0.205712   
                 83         deprel_acl                     ELH -0.365694   
deprel_acl:relcl 26   deprel_acl:relcl  ELH - Critical Inquiry  0.216388   
                 181  deprel_acl:relcl        Critical Inquiry  0.141243   
...                                ...                     ...       ...   
ttr_VERB         28           ttr_VERB        Critical Inquiry  0.582864   
                 84           ttr_VERB  ELH - Critical Inquiry -0.080441   
ttr_mean         11           ttr_mean        Critical Inquiry  0.783724   
                 10           ttr_mean                     ELH  0.736068   
                 93           ttr_mean  ELH - Critical Inquiry  0.047656   

                            raw  feat_rank  index  feat_diff_rank  
feat                                                               
deprel_acl       41    0.687692         42    0.0            42.0  
                 82    9.730251         42    NaN            42.0  
                 83    9.042559         42    NaN            42.0  
deprel_acl:relcl 26    1.066306         91    1.0            27.0  
                 181  14.215942         91    NaN            27.0  
...                         ...        ...    ...             ...  
ttr_VERB         28    0.701367         15    NaN            85.0  
                 84   -0.006918         15  129.0            85.0  
ttr_mean         11    0.330114          6    NaN            94.0  
                 10    0.328145          6    NaN            94.0  
                 93    0.001969          6  130.0            94.0  

[393 rows x 7 columns]

In [154]:
df2 = get_diff_rows(df_smpl_feats)
df2

,feat,target,z,raw,feat_rank,feat_diff_rank
101,pos_POS,ELH - Critical Inquiry,-1.092179,-5.458668,2,1
31,deprel_nmod:poss,ELH - Critical Inquiry,-1.015511,-9.650275,1,2
103,pos_PRP$,ELH - Critical Inquiry,-0.638660,-4.320971,4,3
67,phrase_ROOT,ELH - Critical Inquiry,0.389665,9.741129,25,4
75,phrase_VP,ELH - Critical Inquiry,0.373251,144.777083,28,5
...,...,...,...,...,...,...
105,pos_RBR,ELH - Critical Inquiry,-0.002708,-0.004667,98,127
84,pos_CD,ELH - Critical Inquiry,-0.002021,-0.006908,96,128
96,pos_NN,ELH - Critical Inquiry,-0.000552,-0.012621,100,129
60,phrase_NAC,ELH - Critical Inquiry,0.000000,0.000000,130,130


In [ ]:
def get_info_comparison(groups_train):
    name1,q1 = groups_train[0]
    name2,q2 = groups_train[1]
    df

,feature,weight,mean_Literature,mean_Philosophy,run,comparison
4,deprel_advmod,1.580204,-0.362579,0.220967,4.5,1975-2000 Philosophy vs 1975-2000 Literature
13,deprel_compound,1.490612,-0.306393,0.000601,4.5,1975-2000 Philosophy vs 1975-2000 Literature
29,deprel_mark,1.146667,-0.795709,-0.103632,4.5,1900-1925 Philosophy vs 1900-1925 Literature
12,deprel_ccomp,1.132960,-0.628811,0.501874,4.5,2000-2025 Philosophy vs 2000-2025 Literature
96,pos_NN,1.068158,-0.792170,0.192998,4.5,1900-1925 Philosophy vs 1900-1925 Literature
...,...,...,...,...,...,...
21,deprel_det,-1.001586,0.039470,0.040351,4.5,1950-1975 Philosophy vs 1950-1975 Literature
21,deprel_det,-1.018498,0.137027,-0.051899,4.5,1975-2000 Philosophy vs 1975-2000 Literature
87,pos_FW,-1.144128,0.874235,-0.175874,4.5,1900-1925 Philosophy vs 1900-1925 Literature
21,deprel_det,-1.263030,-0.057937,0.336035,4.5,1925-1950 Philosophy vs 1925-1950 Literature


In [10]:
get_slice_info_df_preds(dfx.index.tolist())

,comparison,predict_type,prob_correct,perc_correct,num_correct,num_runs,support,num_samples
target,,,,,,,,
Literature,1900-1925 Philosophy vs 1900-1925 Literature,cv,0.857448,0.888298,668,10,1504.0,752
Literature,1900-1925 Philosophy vs 1900-1925 Literature,unseen,0.719473,0.741998,10014,10,1504.0,13496
Literature,1925-1950 Philosophy vs 1925-1950 Literature,cv,0.888963,0.914643,1243,10,2000.0,1359
Literature,1925-1950 Philosophy vs 1925-1950 Literature,unseen,0.670708,0.685483,9727,10,2000.0,14190
Literature,1950-1975 Philosophy vs 1950-1975 Literature,cv,0.868743,0.905020,2001,10,2000.0,2211
Literature,1950-1975 Philosophy vs 1950-1975 Literature,unseen,0.816842,0.859530,12244,10,2000.0,14245
Literature,1975-2000 Philosophy vs 1975-2000 Literature,cv,0.871858,0.906886,3253,10,2000.0,3587
Literature,1975-2000 Philosophy vs 1975-2000 Literature,unseen,0.854363,0.902934,12865,10,2000.0,14248
Literature,2000-2025 Philosophy vs 2000-2025 Literature,cv,0.918025,0.949047,5774,10,2000.0,6084


In [11]:

def get_slice_info(slice_id, df_preds=None):
    out_d = {}
    if df_preds is None:
        df_preds = get_df_preds()
    
    return df_preds


In [12]:
for id,x in STASH_SLICES_NLP.items():
    break

In [13]:
get_slice_info(id)

,prob_Literature,prob_Philosophy,support,run,predict_type,comparison
id,,,,,,
lit/456903__06,0.021961,0.978039,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/3713754__02,0.977459,0.022541,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/457155__03,0.996810,0.003190,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/433422__05,0.991444,0.008556,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/456699__05,0.161604,0.838396,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
...,...,...,...,...,...,...
lit/27760277__03,0.979312,0.020688,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature
phil/10.2307/2025409__04,0.000012,0.999988,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature
phil/10.2307/2953730__03,0.010580,0.989420,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature


In [14]:
display_slice_predictions(doc, 'weight_z', 'word')

NameError: name 'display_slice_predictions' is not defined